# 📄 Document Question Answering System (RAG)
### A Retrieval-Augmented Generation Pipeline built with Cohere + Pinecone + LangChain + Streamlit
---

 **Live Demo:** https://document-app-rag-7zrsnhmephfzjq7zbvywwv.streamlit.app/

## 1. Project Overview

This project implements a **Retrieval-Augmented Generation (RAG)** based Document Question Answering System.

The system allows users to upload custom PDF documents. It then:
- Extracts text from the document
- Converts the content into vector embeddings
- Stores the embeddings in a vector database
- Retrieves the most relevant chunks for a user's question
- Generates a **context-aware, grounded answer** using a language model

Unlike a standard LLM chatbot, this approach retrieves relevant information from the uploaded document *before* generating a response — improving factual accuracy and reducing hallucination.

### Objectives
- Build a document ingestion pipeline for custom documents
- Process unstructured text using chunking techniques
- Generate embeddings using a pre-trained embedding model
- Store embeddings in a vector database
- Retrieve relevant document chunks for user queries
- Generate grounded answers using retrieved context

### System Architecture

```
User Uploads Document
        |
        v
PDF Text Extraction
        |
        v
Text Chunking
        |
        v
Cohere Embedding Generation
        |
        v
Pinecone Vector Database
        |
        v
User Query
        |
        v
Similarity Search
        |
        v
Retrieved Context
        |
        v
Cohere Language Model
        |
        v
Final Answer
```

### Technologies Used
| Component | Technology |
|---|---|
| Web App Framework | Streamlit |
| PDF Parsing | PyPDF2 / pypdf |
| Text Chunking | LangChain Text Splitters |
| Embeddings | Cohere `embed-english-v3.0` |
| Vector Database | Pinecone |
| Language Model | Cohere `command-r-08-2024` |

## 2. Environment Setup & Dependencies

Install all the needed packages.

In [2]:
# Install dependencies
!pip install -q "requests==2.32.4" streamlit langchain langchain-community langchain-text-splitters cohere pinecone python-dotenv PyPDF2 pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 910.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 whi

In [3]:
# Core imports used throughout this notebook
import os
from uuid import uuid4
from pathlib import Path

from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from cohere import ClientV2
from pinecone import Pinecone, ServerlessSpec

print("All libraries imported successfully.")

All libraries imported successfully.


## 3. Configuration — API Keys

In [4]:
# Retrieve API keys securely (Colab Secrets preferred, getpass as fallback)
from getpass import getpass

def get_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass(f"Enter value for {name}: ")

COHERE_API_KEY = get_secret("COHERE_API_KEY")
PINECONE_API_KEY = get_secret("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "document-qa-rag")

print("Keys loaded. Pinecone index name:", PINECONE_INDEX_NAME)

Keys loaded. Pinecone index name: document-qa-rag


In [5]:
# Initialize API clients
co = ClientV2(api_key=COHERE_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create the Pinecone index if it doesn't already exist
# (embed-english-v3.0 produces 1024-dimensional embeddings, cosine similarity)
existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Created new Pinecone index: {PINECONE_INDEX_NAME}")
else:
    print(f"Using existing Pinecone index: {PINECONE_INDEX_NAME}")

index = pc.Index(PINECONE_INDEX_NAME)

Using existing Pinecone index: document-qa-rag


## 4. Document Loading


load_documents() scans a data folder and loads the text content of every .pdf and .txt file it finds.

**For this notebook:** upload one or more PDF/TXT files below, or point DATA_FOLDER at a folder in your Google Drive.

In [6]:
# Upload files directly to Colab (equivalent to the project's data/ folder)
from google.colab import files

DATA_FOLDER = "data"
os.makedirs(DATA_FOLDER, exist_ok=True)

print("Upload the PDF/TXT document(s) you want to index (e.g. resume5.pdf):")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = Path(DATA_FOLDER) / filename
    dest.write_bytes(uploaded[filename])
    print(f"Saved -> {dest}")

Upload the PDF/TXT document(s) you want to index (e.g. resume5.pdf):


Saving sde infosys resume.pdf to sde infosys resume.pdf
Saved -> data/sde infosys resume.pdf


In [7]:
def load_documents(data_folder="data"):
    """
    Loads all PDF and TXT files from the given folder.

    Args:
        data_folder (str): Path to the folder containing documents.

    Returns:
        list: List of dictionaries with filename and extracted text.
    """
    documents = []
    folder = Path(data_folder)

    if not folder.exists():
        raise FileNotFoundError(f"Folder '{data_folder}' not found.")

    for file in folder.iterdir():

        # -------- PDF Files --------
        if file.suffix.lower() == ".pdf":
            reader = PdfReader(file)
            text = ""
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
            documents.append({"filename": file.name, "text": text})

        # -------- TXT Files --------
        elif file.suffix.lower() == ".txt":
            with open(file, "r", encoding="utf-8") as f:
                text = f.read()
            documents.append({"filename": file.name, "text": text})

    return documents


documents = load_documents(DATA_FOLDER)

print(f"Documents loaded: {len(documents)}")
for doc in documents:
    print(f" - {doc['filename']}  ({len(doc['text'])} characters)")

Documents loaded: 1
 - sde infosys resume.pdf  (2270 characters)


## 5. Text Chunking

 Long documents are split into overlapping chunks using LangChain's RecursiveCharacterTextSplitter, so each chunk is small enough to embed meaningfully while retaining local context via the overlap.

| Parameter | Value |
|---|---|
| Chunk Size | 500 characters |
| Chunk Overlap | 100 characters |

In [8]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

def chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """
    Split each loaded document's text into overlapping chunks.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = []
    for doc in documents:
        split_text = splitter.split_text(doc["text"])
        for text in split_text:
            chunks.append({"text": text, "source": doc["filename"]})

    return chunks


chunks = chunk_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print("\nSample chunk:")
print(chunks[0] if chunks else "No chunks generated.")

Total chunks created: 6

Sample chunk:
{'text': 'Srinidhi Reddy Almareddy \nEmail :  srinidhireddyyy786@gmail.com  \nLinkedIn :  www.linkedin.com/in/srinidhi-reddy-a-655604354  \nPhone :   +91 9618273065 \nGitHub  : https://github.com/srinidhireddy786  \nCareer Objective : \nA motivated fourth-year Computer Science (AI & ML) student with a strong foundation in Java, Python, Data \nStructures & Algorithms, DBMS, OOP and web development. Passionate about software engineering and', 'source': 'sde infosys resume.pdf'}


## 6. Embedding Generation (Cohere)

Each text chunk is converted into a 1024-dimensional vector using Cohere's embed-english-v3.0 model with input_type="search_document"

In [9]:
def create_embeddings(chunks):
    """
    Generate Cohere embeddings for all chunks.
    """
    texts = [chunk["text"] for chunk in chunks]

    response = co.embed(
        model="embed-english-v3.0",
        input_type="search_document",
        texts=texts,
        embedding_types=["float"]
    )

    return response.embeddings.float


embeddings = create_embeddings(chunks)

print(f"Embeddings generated : {len(embeddings)}")
print(f"Embedding dimension  : {len(embeddings[0])}")

Embeddings generated : 6
Embedding dimension  : 1024


## 7. Vector Storage (Pinecone)

The embeddings are upserted into a Pinecone index. Each vector is stored with:
- A unique UUID as its ID
- The embedding values
- Metadata containing the original chunk text and its source filename

A **namespace** is used so that vectors from different documents/sessions don't collide (mirrors how the Streamlit app namespaces vectors by uploaded filename).

In [10]:
def store_vectors(chunks, embeddings, namespace="colab-demo"):
    """
    Store embeddings in Pinecone under the given namespace.
    """
    vectors = []

    for chunk, embedding in zip(chunks, embeddings):
        vectors.append({
            "id": str(uuid4()),
            "values": embedding,
            "metadata": {
                "text": chunk["text"],
                "source": chunk["source"]
            }
        })

    index.upsert(vectors=vectors, namespace=namespace)
    print(f"Successfully stored {len(vectors)} vectors in Pinecone (namespace='{namespace}').")
    return namespace


NAMESPACE = "colab-demo"
store_vectors(chunks, embeddings, namespace=NAMESPACE)

Successfully stored 6 vectors in Pinecone (namespace='colab-demo').


'colab-demo'

## 8. Retrieval — Similarity Search

The user's question is embedded with input_type="search_query", then Pinecone's query() finds the top_k most similar stored chunks using cosine similarity.

In [12]:
def embed_query(question):
    """
    Convert a user question into an embedding.
    """
    response = co.embed(
        model="embed-english-v3.0",
        input_type="search_query",
        texts=[question],
        embedding_types=["float"]
    )
    return response.embeddings.float[0]


def retrieve_context(question, top_k=3, namespace=None):
    """
    Retrieve the most relevant chunks from Pinecone.
    """
    query_embedding = embed_query(question)

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace=namespace
    )

    return results

## 9. Answer Generation (Cohere Command-R)

The retrieved chunks are concatenated into a context block and passed to `command-r-08-2024`, with an instruction to answer **only** from that context — and to say so explicitly if the answer isn't present.

In [19]:
import time

def generate_answer(question, results):
    """
    Generate an answer using the retrieved context with validation logging.
    """
    start_time = time.time()

    # 1. Validate Retrieval Results
    num_matches = len(results.matches) if results and hasattr(results, 'matches') else 0
    print(f"\n[VALIDATION LOG] Question: '{question}'")
    print(f"[VALIDATION LOG] Retrieved Chunks: {num_matches}")

    if num_matches == 0:
        print("[VALIDATION LOG] Status: ⚠️ No context found in Pinecone.")
        return "I couldn't find that information in the provided document."

    # 2. Build Context
    context = ""
    for match in results.matches:
        context += match.metadata.get("text", "") + "\n\n"

    context_length = len(context.strip())
    print(f"[VALIDATION LOG] Total Context Length: {context_length} characters")

    # 3. Construct Prompt
    prompt = (
        "Answer the question ONLY using the context below.\n\n"
        "If the answer is not available in the context, reply:\n"
        "\"I couldn't find that information in the provided document.\"\n\n"
        f"Context:\n{context}\n"
        f"Question:\n{question}"
    )

    # 4. Query Language Model
    response = co.chat(
        model="command-r-08-2024",
        messages=[{"role": "user", "content": prompt}]
    )

    answer_text = response.message.content[0].text.strip()
    elapsed_time = time.time() - start_time

    # 5. Output Validation Logging
    is_fallback = "I couldn't find that information" in answer_text
    print(f"[VALIDATION LOG] Generation Latency: {elapsed_time:.2f}s")
    print(f"[VALIDATION LOG] Fallback Answer Triggered: {is_fallback}")
    print("[VALIDATION LOG] Status: ✅ Answer successfully generated.\n")

    return answer_text

## 10. End-to-End Pipeline Demo

Ask a question about the document(s) you uploaded in Section 4. This ties together retrieval + generation, and also prints the retrieved chunks with their similarity scores for validation.


In [20]:
question = input("Enter your question about the uploaded document: ")

results = retrieve_context(question, top_k=3, namespace=NAMESPACE)

answer = generate_answer(question, results)

print("\n===================================== ANSWER ==========================================================\n")
print(answer)

print("\n======================================== RETRIEVAL VALIDATION ==========================================\n")
for i, match in enumerate(results.matches, start=1):
    print(f"Result {i}")
    print(f"Similarity Score : {match.score:.4f}")
    print(f"Source           : {match.metadata.get('source', 'N/A')}")
    print(f"Text             : {match.metadata['text'][:200]}...")
    print("-" * 60)


Enter your question about the uploaded document: what is srinidhi cgpa

[VALIDATION LOG] Question: 'what is srinidhi cgpa'
[VALIDATION LOG] Retrieved Chunks: 3
[VALIDATION LOG] Total Context Length: 1356 characters
[VALIDATION LOG] Generation Latency: 0.52s
[VALIDATION LOG] Fallback Answer Triggered: False
[VALIDATION LOG] Status: ✅ Answer successfully generated.


===================================== ANSWER ==========================================================

Srinidhi's CGPA is 8.59.

======================================== RETRIEVAL VALIDATION ==========================================

Result 1
Similarity Score : 0.5159
Source           : sde infosys resume.pdf
Text             : Srinidhi Reddy Almareddy 
Email :  srinidhireddyyy786@gmail.com  
LinkedIn :  www.linkedin.com/in/srinidhi-reddy-a-655604354  
Phone :   +91 9618273065 
GitHub  : https://github.com/srinidhireddy786  ...
------------------------------------------------------------
Result 2
Similarity Score : 0.3811

## 11. System Metrics Summary

Summary of the configuration used throughout this pipeline:

| Metric | Value |
|---|---|
| Embedding Model | Cohere `embed-english-v3.0` |
| Embedding Dimension | 1024 |
| Vector Database | Pinecone |
| Similarity Metric | Cosine |
| Language Model | Cohere `command-r-08-2024` |
| Chunking Method | Recursive Character Text Splitter |
| Chunk Size | 500 |
| Chunk Overlap | 100 |
| Supported Document Types | PDF, TXT |

## 13. System Optimization Experiments

To improve our pipeline's accuracy, we tested three optimization strategies:

a — Chunk Size Tuning: Evaluated how different chunk sizes and overlaps impact retrieval scores for the same question.

b — Hybrid Search (BM25 + Vector): Combined keyword matching with vector search so the system better captures exact terms like names, numbers, and codes.

c — Cohere Re-Ranking: Added Cohere’s rerank model as a second pass to re-score retrieved chunks and boost the most relevant results to the top.

In [21]:
# ---- 13a. Chunk Boundary Experiment ----
TEST_QUESTION = "What programming languages are mentioned in the document?"

chunk_configs = [
    {"chunk_size": 250, "chunk_overlap": 50},
    {"chunk_size": 500, "chunk_overlap": 100},  # Default
    {"chunk_size": 800, "chunk_overlap": 150},
]

print("========== CHUNK BOUNDARY EXPERIMENT ==========\n")

for cfg in chunk_configs:
    ns = f"exp-{cfg['chunk_size']}-{cfg['chunk_overlap']}"
    exp_chunks = chunk_documents(documents, cfg["chunk_size"], cfg["chunk_overlap"])

    # Generate embeddings and store in Pinecone namespace
    exp_embeddings = create_embeddings(exp_chunks)
    store_vectors(exp_chunks, exp_embeddings, namespace=ns)

    # Retrieve & evaluate top match
    results = retrieve_context(TEST_QUESTION, top_k=3, namespace=ns)
    top_score = results.matches[0].score if results.matches else 0.0

    print(f"Size: {cfg['chunk_size']} | Overlap: {cfg['chunk_overlap']} | Total Chunks: {len(exp_chunks)} | Top Score: {top_score:.4f}")

print(f"\nQuestion: {TEST_QUESTION}")

========== CHUNK BOUNDARY EXPERIMENT ==========

Successfully stored 11 vectors in Pinecone (namespace='exp-250-50').
Size: 250 | Overlap: 50 | Total Chunks: 11 | Top Score: 0.3446
Successfully stored 6 vectors in Pinecone (namespace='exp-500-100').
Size: 500 | Overlap: 100 | Total Chunks: 6 | Top Score: 0.3457
Successfully stored 4 vectors in Pinecone (namespace='exp-800-150').
Size: 800 | Overlap: 150 | Total Chunks: 4 | Top Score: 0.2967

Question: What programming languages are mentioned in the document?


Retrieval Strategy Analysis Insights:

Smaller Chunks (e.g., 250): Tend to raise top-1 precision on narrow factual questions, but can lose surrounding context.

Larger Chunks (e.g., 800): Preserve broader document context, but can dilute the embedding signal and reduce similarity scores.

Takeaway: Select the chunking configuration with the highest top score (top_score) for questions representative of your specific use case.

#Hybrid Search: Keyword (BM25) + Vector

In [24]:
!pip install -q rank_bm25
import numpy as np
from rank_bm25 import BM25Okapi

def hybrid_search(question, chunks, embeddings, top_k=3, vector_weight=0.6):
    """
    Combines BM25 keyword matching with vector cosine similarity.
    """
    # 1. BM25 Keyword Scoring
    tokenized_corpus = [c["text"].lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_scores = bm25.get_scores(question.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1.0
    norm_bm25 = [s / max_bm25 for s in bm25_scores]

    # 2. Vector Cosine Similarity
    query_vec = np.array(embed_query(question))
    chunk_vecs = np.array(embeddings)
    cosine_scores = (chunk_vecs @ query_vec) / (
        np.linalg.norm(chunk_vecs, axis=1) * np.linalg.norm(query_vec) + 1e-8
    )

    # 3. Blend Scores
    ranked = sorted(
        zip(chunks, cosine_scores, norm_bm25),
        key=lambda x: (vector_weight * x[1] + (1 - vector_weight) * x[2]),
        reverse=True
    )[:top_k]

    # 4. Print Results
    print(f"========== HYBRID SEARCH RESULTS: '{question}' ==========\n")
    for i, (chunk, vec_score, kw_score) in enumerate(ranked, start=1):
        blended = vector_weight * vec_score + (1 - vector_weight) * kw_score
        print(f"Result {i} | Blended: {blended:.4f} | Vector: {vec_score:.4f} | Keyword: {kw_score:.4f}")
        print(f"   {chunk['text'][:150]}...")
        print("-" * 60)

    return ranked

# Run Hybrid Search using default chunks & embeddings
_ = hybrid_search(TEST_QUESTION, chunks, embeddings, top_k=3, vector_weight=0.6)

========== HYBRID SEARCH RESULTS: 'What programming languages are mentioned in the document?' ==========

Result 1 | Blended: 0.5578 | Vector: 0.2631 | Keyword: 1.0000
   Graduating Year : 2027 | CGPA:8.59 
 Intermediate Education (M.P.C) 
Narayana Junior college 
Graduating year : 2023  |  Percentage:97.9% 
 Secondar...
------------------------------------------------------------
Result 2 | Blended: 0.3088 | Vector: 0.2948 | Keyword: 0.3299
   Srinidhi Reddy Almareddy 
Email :  srinidhireddyyy786@gmail.com  
LinkedIn :  www.linkedin.com/in/srinidhi-reddy-a-655604354  
Phone :   +91 961827306...
------------------------------------------------------------
Result 3 | Blended: 0.2990 | Vector: 0.3017 | Keyword: 0.2950
   Structures & Algorithms, DBMS, OOP and web development. Passionate about software engineering and 
problem solving, with hands-on experience in buildi...
------------------------------------------------------------


#c. Re-Ranking Layer (Cohere Rerank)

In [26]:
# ---- 13c. Re-Ranking Layer (Cohere Rerank) ----

def retrieve_with_rerank(question, namespace=NAMESPACE, initial_top_k=10, final_top_k=3):
    """
    Retrieves a wider candidate list via vector search, then re-ranks
    them using Cohere's rerank model for precise relevance.
    """
    # 1. Retrieve top candidates from Pinecone
    initial_results = retrieve_context(question, top_k=initial_top_k, namespace=namespace)
    candidates = [m.metadata["text"] for m in initial_results.matches]

    if not candidates:
        return initial_results

    # 2. Re-rank candidates with Cohere
    rerank_response = co.rerank(
        model="rerank-english-v3.0",
        query=question,
        documents=candidates,
        top_n=final_top_k
    )

    # 3. Print Results
    print(f"========== RE-RANKED RESULTS: '{question}' ==========\n")
    for rank, result in enumerate(rerank_response.results, start=1):
        orig_match = initial_results.matches[result.index]
        print(f"Rank {rank} | Rerank Score: {result.relevance_score:.4f} | Vector Score: {orig_match.score:.4f}")
        print(f"   {candidates[result.index][:150]}...")
        print("-" * 60)

    return rerank_response

# Run Re-Ranking test
_ = retrieve_with_rerank(TEST_QUESTION)
print("\nRe-ranking is especially useful when top_k for vector search is set high (wide recall net)")
print("and you need the final few chunks passed to the LLM to be as precise as possible.")

========== RE-RANKED RESULTS: 'What programming languages are mentioned in the document?' ==========

Rank 1 | Rerank Score: 0.1527 | Vector Score: 0.2636
   Graduating Year : 2027 | CGPA:8.59 
 Intermediate Education (M.P.C) 
Narayana Junior college 
Graduating year : 2023  |  Percentage:97.9% 
 Secondar...
------------------------------------------------------------
Rank 2 | Rerank Score: 0.0715 | Vector Score: 0.3452
   Core Cs                                        : DSA, OOP, DBMS, OS 
Frameworks & Tools                   : Git, GitHub, VS Code 
Projects: 
 MyStudy...
------------------------------------------------------------
Rank 3 | Rerank Score: 0.0208 | Vector Score: 0.3040
    Integrated automated reminders to improve productivity and time management. 
 Designed and managed a MySQL database for efficient data storage and ...
------------------------------------------------------------

Re-ranking is especially useful when top_k for vector search is set high (wide recal

## 14. Conclusion

This is the complete workflow of a Retrieval-Augmented Generation system:

1. **Ingestion** — loading PDF/TXT documents
2. **Processing** — chunking text with overlap for context preservation
3. **Embedding** — converting chunks into dense vector representations with Cohere
4. **Storage** — indexing vectors in Pinecone for fast similarity search
5. **Retrieval** — finding the most relevant chunks for a given question
6. **Generation** — producing a grounded, context-aware answer with Cohere's Command-R model